## Logs

A detached container's output doesn't vanish — the daemon captures whatever its main process writes to **stdout and stderr**, and `docker logs` reads it back. (That's *why* a containerised app should log to stdout, not to a file inside the container: Docker is already capturing the stream.)

The command and the flags you'll actually use:

- **`docker logs web`** — everything since the container started.
- **`docker logs -f web`** — *follow* new output live (`tail -f` style); Ctrl-C just detaches.
- **`docker logs --tail=100 web`** — the last 100 lines (great before `-f` on a chatty service).
- **`docker logs --since=10m web`** — only the last 10 minutes; also accepts `1h` or a timestamp. Pair with **`--until`** to window a range.
- **`docker logs -t web`** — prefix each line with the daemon's timestamp.

**Where logs actually live.** The daemon hands the stream to the active **logging driver**. The default, **`json-file`**, writes to `/var/lib/docker/containers/<id>/<id>-json.log`, and `docker logs` reads from there.

Two consequences worth knowing now:

- **`docker logs` only works with the `json-file` (default) and `journald` drivers.** Switch to `syslog`, `awslogs`, `fluentd`, `splunk`, etc. and the output goes straight to that system — `docker logs` returns nothing, you read it at the destination.
- **`json-file` logs grow unbounded by default** and can fill a disk. In production you cap them with `--log-opt max-size=10m --log-opt max-file=3` (per-container, or as a daemon default — module 10).